In [6]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

from ingestion import get_transcripts_dataframe, build_index
openai_client = OpenAI()

## Ingestion

In [7]:
documents_df = get_transcripts_dataframe()
index = build_index(documents_df)

## RAG Flow

In [13]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [5]:
print(assistant.rag("Which techniques can help improve self-awareness?"))

Techniques that can help improve self-awareness include:
- Recording yourself and reviewing your gestures, posture, tone, and speaking habits
- Asking yourself what causes your stress, who triggers it, and when it happens
- Questioning your emotions, such as why you feel afraid, guilty, sad, angry, or irritated, and what caused them

https://www.youtube.com/watch?v=ln8rIBZbWAE


## Evaluation

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [8]:
documents_list = []

for doc in documents_df:
    documents_list.append(doc)

In [9]:
doc_idx = {}

for doc in documents_list:
    doc_idx[doc["id"]] = doc

In [ ]:
INSTRUCTIONS_EVALUATION = """
You are a helpful assistant that answers questions regarding self-awareness based on the provided context.

Use the context to find relevant information and provide accurate
answers. 
If the answer is not found in the context,
respond with "I don't know."
"""

assistant_evaluation = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=INSTRUCTIONS_EVALUATION,
)

In [34]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant_evaluation.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [28]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [35]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/500 [00:00<?, ?it/s]

In [37]:
df_results = pd.DataFrame(results)

In [38]:
df_results.head()

,question,answer_llm,answer_orig,document
0,What does emotional intelligence mean in simpl...,Emotional intelligence means how well you hand...,Emotional intelligence is how effectively you ...,0
1,How does emotional intelligence affect the way...,Emotional intelligence helps you handle yourse...,Emotional intelligence is how effectively you ...,0
2,Why is emotional intelligence important in rel...,Emotional intelligence is important in relatio...,Emotional intelligence is how effectively you ...,0
3,What skills are included in emotional intellig...,The four main domains of emotional intelligenc...,Emotional intelligence is how effectively you ...,0
4,How would you explain emotional intelligence t...,Emotional intelligence is how effectively you ...,Emotional intelligence is how effectively you ...,0


In [39]:
df_results.to_csv("data/rag-answers.csv", index=False)